# Этап A: синтетические запросы и эмбеддинги корпуса

Перед запуском: подключить датасет `ru-fin-retrieval` через **+ Add Input**,
меню **Settings** → **Accelerator** → **GPU T4 ×2**; там же проверить,
что интернет включён (пункт читается как **Turn off internet**).

Порядок действий целиком — в `kaggle/README.md` проекта.


## 1. Проверка видеокарты

Должна быть видна Tesla T4. Если команда не найдена — ускоритель не включён.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Библиотеки

`bitsandbytes` нужен для четырёхбитного квантования модели-генератора:
в обычной точности она не поместилась бы в память карты.


In [ ]:
!pip install -q -U sentence-transformers bitsandbytes accelerate


## 3. Код на место


In [ ]:
!cp -r /kaggle/input/ru-fin-retrieval/* /kaggle/working/
%cd /kaggle/working
!ls


## 4. Пробный прогон

Двадцать актов и одна лёгкая модель: проверяем, что датасет виден,
код импортируется и модели скачиваются. Пара минут.


In [ ]:
!python run_phase_a.py \
    --acts /kaggle/input/ru-fin-retrieval/acts.jsonl.gz \
    --limit-acts 20 --skip-queries \
    --headline-models e5-small --ablation-configs \
    --out /kaggle/working/test


## 5. Полный прогон

Около часа. Сначала генерация полутора сотен вопросов, потом эмбеддинги
пяти моделей по базовой нарезке и абляционных нарезок на e5-small.

Посчитанное не пересчитывается: если сессия оборвётся, просто запустите
ячейку заново.


In [ ]:
!python run_phase_a.py --acts /kaggle/input/ru-fin-retrieval/acts.jsonl.gz


## 6. Что получилось


In [ ]:
import json
r = json.load(open('/kaggle/working/report_phase_a.json'))
q = r.get('queries')
if q:
    print(f"запросов принято {q['accepted']}, отклонено {q['rejected']}")
    print('причины отказа:', q['reasons'])
for e in r.get('embeddings', []):
    if 'error' in e:
        print('СБОЙ', e['config'], e['model'], e['error'][:120])
    else:
        print(f"{e['chunks_config']:<12} {e['model']:<12} {e['chunks']:>7} фрагм. "
              f"dim={e['dim']:<5} {e['per_second']:>7} фрагм./с {e['size_mb']:>8} МБ")


## 7. Сохранить и скачать

**Save Version** → **Save & Run All (Commit)**. Без этого результат
пропадёт при закрытии сессии.

Скачать нужно только `queries/synthetic.jsonl` — он маленький. Матрицы
эмбеддингов остаются здесь, их прочитает этап B.
